In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import pipelines as dp

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

In [0]:
df = spark.read.table(SILVER)

In [0]:
@dp.materialized_view(name=f"dim_vehicle")
def dim_vehicle():
    return (
        df.select("vehicleId","vehicleCode","vehicleService","transportationType" "vehicleCharacteristics","brand", "model", "productionYear", "length", "seats", "standingPlaces", "floorHeight", "driveType", "carrier", "airConditioning", "wheelchairsRamp","ticketMachine", "usb" )
        .dropDuplicates(["vehicleId"])
        .withColumn("vehicle_key", F.xxhash64("vehicleId") )
    )

In [0]:
@dp.materialized_view(name="dim_route")
def dim_route():
    return(
        df.select("routeId", "routeShortName", "route_type", "route_color", "route_text_color")
        .dropDuplicates(["routeId"])
        .withColumn("route_key", F.xxhash64("routeId"))
    )

In [0]:
@dp.materialized_view(name="dim_destination")
def dim_route():
    return(
        df.select("headsign")
        .filter(F.col("headsign").isNotNull())
        .dropDuplicates(["headsign"])
        .withColumn("destination_key", F.xxhash64("headsign"))
    )

In [0]:
@dp.materialized_view(name="dim_time")
def dim_route():
    return(
        df.filter(F.col("event_time_local").isNotNull())
        .select(F.to_date("event_time_local").alias("date"),
                F.year("event_time_local").alias("year"),
                F.month("event_time_local").alias("month"),
                F.dayofmonth("event_time_local").alias("day"),
                F.dayofweek("event_time_local").alias("day_of_week")
                F.hour("event_time_local").alias("hour"))
        .dropDuplicates(["date", "hour"])
        .withColumn("is_weekend", F.col("day_in_week").isin(1,7))
        .withColumn("time_key", F.xxhash64("date", "hour"))
    )